## Aggregations

* The simplest grouping is to just summarize a complete DataFrame by performing an aggregation in a select statement.
* A “group by” allows you to specify one or more keys as well as one or more aggregation functions to transform the value columns.
* A “window” gives you the ability to specify one or more keys as well as one or more aggregation functions to transform the value columns. However, the rows input to the function are somehow related to the current row.
* A “grouping set,” which you can use to aggregate at multiple different levels. Grouping sets are available as a primitive in SQL and via rollups and cubes in DataFrames.
* A “rollup” makes it possible for you to specify one or more keys as well as one or more aggregation functions to transform the value columns, which will be summarized hierarchically.
* A “cube” allows you to specify one or more keys as well as one or more aggregation functions to transform the value columns, which will be summarized across all combinations of columns.


In [1]:
import sys
import os

spark_home = "/opt/spark"
sys.path.append(os.path.join(spark_home, "python"))
sys.path.append(os.path.join(spark_home, "python/lib/py4j-0.10.9.7-src.zip"))

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Aggregations").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 15:24:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.format('csv').option('header','true').option('inferSchema', 'true').load('data/customers.csv')
df.cache()
df.createOrReplaceTempView("dfTable")

In [3]:
df.show(5)
df = df.limit(10000).select("Index", "Customer Id", "First Name", "Last Name", "Company", "City", "Country", "Email", "Subscription Date")

+-----+---------------+----------+----------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|Index|    Customer Id|First Name| Last Name|             Company|          City|             Country|             Phone 1|             Phone 2|               Email|Subscription Date|             Website|
+-----+---------------+----------+----------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|    1|ffeCAb7AbcB0f07|     Jared|    Jarvis|    Sanchez-Fletcher| Hatfieldshire|             Eritrea|  274.188.8773x41185|001-215-760-4642x969|gabriellehartman@...|       2021-11-11|https://www.mccar...|
|    2|b687FfC4F1600eC|     Marie|    Malone|           Mckay PLC|Robertsonburgh|            Botswana|        283-236-9529| (189)129-8356x63741|kstafford@sexton.com|       2021-05-

In [4]:
df.show()

+-----+---------------+----------+----------+--------------------+-----------------+--------------------+--------------------+-----------------+
|Index|    Customer Id|First Name| Last Name|             Company|             City|             Country|               Email|Subscription Date|
+-----+---------------+----------+----------+--------------------+-----------------+--------------------+--------------------+-----------------+
|    1|ffeCAb7AbcB0f07|     Jared|    Jarvis|    Sanchez-Fletcher|    Hatfieldshire|             Eritrea|gabriellehartman@...|       2021-11-11|
|    2|b687FfC4F1600eC|     Marie|    Malone|           Mckay PLC|   Robertsonburgh|            Botswana|kstafford@sexton.com|       2021-05-14|
|    3|9FF9ACbc69dcF9c|    Elijah|   Barrera|      Marks and Sons|          Kimbury|            Barbados|jeanettecross@bro...|       2021-03-17|
|    4|b49edDB1295FF6E|    Sheryl|Montgomery|Kirby, Vaughn and...|      Briannaview|Antarctica (the t...|thomassierra@barr...|    

In [5]:
## Count

df.count()

10000

In [6]:
from pyspark.sql.functions import count

df.select( count("City") ).show()
df.select( count("*") ).show()

+-----------+
|count(City)|
+-----------+
|      10000|
+-----------+

+--------+
|count(1)|
+--------+
|   10000|
+--------+



In [7]:
from pyspark.sql.functions import countDistinct

df.select( countDistinct("Country") ).show()

+-----------------------+
|count(DISTINCT Country)|
+-----------------------+
|                    243|
+-----------------------+



In [8]:
from pyspark.sql.functions import approx_count_distinct

df.select( approx_count_distinct( "Country", 0.1 ) ).show()

[Stage 18:>                                                         (0 + 1) / 1]

+------------------------------+
|approx_count_distinct(Country)|
+------------------------------+
|                           204|
+------------------------------+



In [9]:
from pyspark.sql.functions import first, last

df.select( first("City"), last("Country") ).show()

+-------------+-------------+
|  first(City)|last(Country)|
+-------------+-------------+
|Hatfieldshire|        China|
+-------------+-------------+



In [10]:
from pyspark.sql.functions import min, max

df.select( min("City"), max("Country") ).show()

+---------+------------+
|min(City)|max(Country)|
+---------+------------+
|Aaronberg|    Zimbabwe|
+---------+------------+



In [11]:
from pyspark.sql.functions import sum

df.select( sum("Index") ).show()

+----------+
|sum(Index)|
+----------+
|  50005000|
+----------+



In [12]:
from pyspark.sql.functions import sum_distinct

df.select( sum_distinct("Index") ).show()

+-------------------+
|sum(DISTINCT Index)|
+-------------------+
|           50005000|
+-------------------+



In [13]:
from pyspark.sql.functions import avg, count, sum, expr

df.select(
    count("Index").alias("total_transactions"),
    sum("Index").alias("total_purchases"),
    avg("Index").alias("avg_purchases"),
    expr("mean(Index)").alias("mean_purchases")
).selectExpr(
    "total_transactions",
    "total_purchases",
    "total_purchases/total_transactions",
    "avg_purchases",
    "mean_purchases"
).show()

+------------------+---------------+--------------------------------------+-------------+--------------+
|total_transactions|total_purchases|(total_purchases / total_transactions)|avg_purchases|mean_purchases|
+------------------+---------------+--------------------------------------+-------------+--------------+
|             10000|       50005000|                                5000.5|       5000.5|        5000.5|
+------------------+---------------+--------------------------------------+-------------+--------------+



In [14]:
## Variance and Standars Desviation

from pyspark.sql.functions import var_pop, stddev_pop
from pyspark.sql.functions import var_samp, stddev_samp

df.select( var_pop("Index"), var_samp("Index"), stddev_pop("Index"), stddev_samp("Index") ).show()

+--------------+-----------------+-----------------+------------------+
|var_pop(Index)|  var_samp(Index)|stddev_pop(Index)|stddev_samp(Index)|
+--------------+-----------------+-----------------+------------------+
|    8333333.25|8334166.666666667|2886.751331514372|2886.8956799071675|
+--------------+-----------------+-----------------+------------------+



In [15]:
## Skewness and Kurtosis
from pyspark.sql.functions import skewness, kurtosis

df.select( skewness("Index"), kurtosis("Index") ).show()

+---------------+-----------------+
|skewness(Index)|  kurtosis(Index)|
+---------------+-----------------+
|            0.0|-1.20000002399999|
+---------------+-----------------+



In [16]:
## Covariance and Correlation

from pyspark.sql.functions import corr, covar_pop, covar_samp

df.select(
    corr("Index", "Index"),
    covar_samp("Index", "Index"),
    covar_pop("Index", "Index"),
).show()

+------------------+------------------------+-----------------------+
|corr(Index, Index)|covar_samp(Index, Index)|covar_pop(Index, Index)|
+------------------+------------------------+-----------------------+
|               1.0|       8334166.666666667|             8333333.25|
+------------------+------------------------+-----------------------+



In [17]:
## Aggreation to complex types

from pyspark.sql.functions import collect_set, collect_list

df.agg( collect_set("Country"), collect_list("Country") ).show()


+--------------------+---------------------+
|collect_set(Country)|collect_list(Country)|
+--------------------+---------------------+
|[Finland, Svalbar...| [Eritrea, Botswan...|
+--------------------+---------------------+



### Grouping

In [18]:
df.groupBy("Country", "City").count().sort("count").show()

+--------------------+-----------------+-----+
|             Country|             City|count|
+--------------------+-----------------+-----+
|             Eritrea|    Hatfieldshire|    1|
|            Botswana|   Robertsonburgh|    1|
|            Barbados|          Kimbury|    1|
|Antarctica (the t...|      Briannaview|    1|
|          Micronesia|    South Brianna|    1|
|              Panama|         Reidtown|    1|
|United States of ...|     Mcintoshberg|    1|
|      American Samoa|   Lake Pedrofort|    1|
|Saint Vincent and...|       Petersbury|    1|
|            Tanzania|         Hillside|    1|
|Central African R...|  New Brookemouth|    1|
|Falkland Islands ...|        Jessestad|    1|
|               Sudan|     South Travis|    1|
|       Faroe Islands|       Joannaport|    1|
|          Bangladesh|       Port Heidi|    1|
|             Estonia|    Port Jadebury|    1|
|           Venezuela|    Mercedesville|    1|
|               Nauru|      Makaylaport|    1|
|Saint Kitts 

In [19]:
df.groupBy("Country").agg(
    count("City").alias("city_num"),
    expr("count(City)")
).show(5)

+--------------------+--------+-----------+
|             Country|city_num|count(City)|
+--------------------+--------+-----------+
|             Eritrea|      43|         43|
|            Botswana|      46|         46|
|            Barbados|      45|         45|
|Antarctica (the t...|      57|         57|
|          Micronesia|      44|         44|
+--------------------+--------+-----------+
only showing top 5 rows



In [20]:
df.groupBy("Country").agg(
    expr("count(City)"),
    expr("cast(avg(Index) as int)")
).show(10)

+--------------------+-----------+-----------------------+
|             Country|count(City)|CAST(avg(Index) AS INT)|
+--------------------+-----------+-----------------------+
|             Eritrea|         43|                   4736|
|            Botswana|         46|                   5173|
|            Barbados|         45|                   5135|
|Antarctica (the t...|         57|                   5147|
|          Micronesia|         44|                   5180|
|              Panama|         51|                   5092|
|United States of ...|         45|                   4789|
|      American Samoa|         38|                   5269|
|Saint Vincent and...|         41|                   4596|
|            Tanzania|         36|                   4732|
+--------------------+-----------+-----------------------+
only showing top 10 rows



### Window functions

In [21]:
from pyspark.sql.functions import col, to_date

dfWithDate = df.withColumn("date", to_date( col("Subscription Date"), "yyyy-MM-d" ))
dfWithDate.show()
# df.show()

+-----+---------------+----------+----------+--------------------+-----------------+--------------------+--------------------+-----------------+----------+
|Index|    Customer Id|First Name| Last Name|             Company|             City|             Country|               Email|Subscription Date|      date|
+-----+---------------+----------+----------+--------------------+-----------------+--------------------+--------------------+-----------------+----------+
|    1|ffeCAb7AbcB0f07|     Jared|    Jarvis|    Sanchez-Fletcher|    Hatfieldshire|             Eritrea|gabriellehartman@...|       2021-11-11|2021-11-11|
|    2|b687FfC4F1600eC|     Marie|    Malone|           Mckay PLC|   Robertsonburgh|            Botswana|kstafford@sexton.com|       2021-05-14|2021-05-14|
|    3|9FF9ACbc69dcF9c|    Elijah|   Barrera|      Marks and Sons|          Kimbury|            Barbados|jeanettecross@bro...|       2021-03-17|2021-03-17|
|    4|b49edDB1295FF6E|    Sheryl|Montgomery|Kirby, Vaughn and..